In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -U huggingface_hub transformers peft accelerate safetensors -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 13.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 94.9 MB/s eta 0:00:00:00:01


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("HF login ok")

HF login ok


In [5]:
import gc
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "google/gemma-4-E4B-it"
ADAPTER_REPO = "raqibcodes/c2c-checkpoints"
ADAPTER_SUBFOLDER = ""  # set "last-checkpoint" only if you explicitly want that snapshot
OUT_DIR = Path("/kaggle/working/c2c_gemma4_e4b_it_fused")
OUT_DIR.mkdir(parents=True, exist_ok=True)

tok_kwargs = {"token": hf_token}
if ADAPTER_SUBFOLDER:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, subfolder=ADAPTER_SUBFOLDER, **tok_kwargs)
else:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_REPO, **tok_kwargs)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",  # avoids burning GPU quota for one-time fuse
    low_cpu_mem_usage=True,
    token=hf_token,
)

if ADAPTER_SUBFOLDER:
    peft_model = PeftModel.from_pretrained(base, ADAPTER_REPO, subfolder=ADAPTER_SUBFOLDER, token=hf_token)
else:
    peft_model = PeftModel.from_pretrained(base, ADAPTER_REPO, token=hf_token)

fused = peft_model.merge_and_unload()
fused.save_pretrained(str(OUT_DIR), safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(str(OUT_DIR))

del peft_model, base, fused
gc.collect()

print("Fused model saved to:", OUT_DIR)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/147M [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]

Fused model saved to: /kaggle/working/c2c_gemma4_e4b_it_fused


In [6]:
from huggingface_hub import HfApi

FUSED_REPO = "raqibcodes/c2c-gemma4-e4b-it-fused"  # change if needed

api = HfApi(token=hf_token)
api.create_repo(repo_id=FUSED_REPO, repo_type="model", private=True, exist_ok=True)
commit = api.upload_folder(
    folder_path="/kaggle/working/c2c_gemma4_e4b_it_fused",
    repo_id=FUSED_REPO,
    repo_type="model",
)
print("Uploaded:", f"https://huggingface.co/{FUSED_REPO}")
print(commit)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: https://huggingface.co/raqibcodes/c2c-gemma4-e4b-it-fused
https://huggingface.co/raqibcodes/c2c-gemma4-e4b-it-fused/commit/fd7d8efc61a9ccddc358daf1a4c20199a2e75d79
